In [ ]:
!pip -q install astroquery astropy matplotlib

In [ ]:

import astropy.units as u
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from astroquery.ipac.irsa import Irsa
from astropy.utils.data import conf
conf.remote_timeout = 120  # los archivos de SPHEREx tardan en responder

In [ ]:
# R Draconis
coord = SkyCoord(ra=248.1675 * u.degree, dec=66.7550 * u.degree, frame="icrs")
size = 0.1  # grados

res = Irsa.query_sia(pos=(coord, 1 * u.arcsec),
                     collection=["spherex_dr1", "spherex_qr3"])
tabla = res.to_table() if hasattr(res, "to_table") else res
print("Imágenes encontradas:", len(tabla))
print(tabla.colnames)

In [ ]:
import time, urllib.error, http.client

tabla.sort("t_min")

# Guardar la lista de imágenes: así no dependen de que la búsqueda vuelva a responder
tabla["access_url", "t_min", "energy_bandpassname", "obs_collection"].write(
    "r_dra_tabla.csv", overwrite=True)

# Ver cómo se llaman los detectores y quedarse con uno
print("Detectores:", set(tabla["energy_bandpassname"]))
detector = "SPHEREx-D2"
sub = tabla[tabla["energy_bandpassname"] == detector]
print(f"Imágenes en {detector}:", len(sub))

def bajar(url, intentos=4):
    """Descarga un recorte; reintenta con espera creciente si IRSA falla."""
    for k in range(intentos):
        try:
            with fits.open(url, cache=True) as hdul:
                return hdul[1].data.copy()
        except (urllib.error.URLError, TimeoutError, OSError,
                http.client.IncompleteRead, http.client.RemoteDisconnected) as e:
            print(f"  fallo ({e}); reintento {k + 1}")
            time.sleep(5 * (k + 1))
    return None

t0 = time.time()
guardados = 0
for i, row in enumerate(sub[:20]):
    url = f"{row['access_url']}?center={coord.ra.deg},{coord.dec.deg}d&size={size}"
    img = bajar(url)
    if img is None:
        print(f"Cuadro {i} omitido")
        continue
    vmin, vmax = ZScaleInterval().get_limits(img)
    plt.figure(figsize=(4, 4))
    plt.imshow(img, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    plt.title(f"MJD {row['t_min']:.2f}")
    plt.axis("off")
    plt.savefig(f"frame_{i:02d}.png", bbox_inches="tight")
    plt.close()
    guardados += 1
    time.sleep(1)  # no martillar un servidor que ya está saturado

print(f"{guardados} cuadros en {time.time() - t0:.0f} s")

In [ ]:
!pip -q install reproject imageio

In [ ]:


import numpy as np, imageio
from astropy.wcs import WCS
from reproject import reproject_interp

# Grilla común: norte arriba, centrada en el objetivo, 6.15"/píxel (escala de SPHEREx)
escala = 6.15 / 3600
npix = int(size / escala)
destino = WCS(naxis=2)
destino.wcs.ctype = ["RA---TAN", "DEC--TAN"]
destino.wcs.crval = [coord.ra.deg, coord.dec.deg]
destino.wcs.crpix = [npix / 2 + 0.5, npix / 2 + 0.5]
destino.wcs.cdelt = [-escala, escala]

cuadros = []
for i, row in enumerate(sub[:20]):
    url = f"{row['access_url']}?center={coord.ra.deg},{coord.dec.deg}d&size={size}"
    try:
        with fits.open(url, cache=True) as hdul:
            datos = hdul[1].data.astype(float)
            wcs_origen = WCS(hdul[1].header)
    except Exception as e:
        print(f"Cuadro {i} omitido: {e}")
        continue
    alineado, _ = reproject_interp((datos, wcs_origen), destino, shape_out=(npix, npix))
    alineado -= np.nanmedian(alineado)                  # quitar el fondo de cielo
    lo, hi = np.nanpercentile(alineado, [5, 99.5])      # contraste por cuadro
    img8 = np.clip((alineado - lo) / (hi - lo), 0, 1)
    img8 = np.nan_to_num(img8) * 255
    cuadros.append(np.flipud(img8).astype(np.uint8))    # flip: norte arriba en el GIF

imageio.mimsave("r_dra.gif", cuadros, duration=0.4, loop=0)
print(len(cuadros), "cuadros alineados")

from IPython.display import Image
Image("r_dra.gif")

In [ ]:
from astropy.time import Time

# ra, dec (grados), movimiento propio en RA y Dec (milisegundos de arco/año), época de la posición
OBJETIVOS = {
    "r_dra":    dict(ra=248.1675,  dec=66.7550,   pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "rw_aur":   dict(ra=76.95642,  dec=30.40142,  pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "barnard":  dict(ra=269.45208, dec=4.69339,   pmra=-802,  pmdec=10362, epoca=2000.0, size=0.1),
    "wise0855": dict(ra=133.79513, dec=-7.24515,  pmra=-8120, pmdec=680,   epoca=2010.5, size=0.1),
    "eps_ind":  dict(ra=331.0438,  dec=-56.7827,  pmra=3960,  pmdec=-2540, epoca=2000.0, size=0.2),
}

def posicion_actual(clave, fecha="2026-01-01"):
    o = OBJETIVOS[clave]
    c = SkyCoord(ra=o["ra"] * u.deg, dec=o["dec"] * u.deg,
                 pm_ra_cosdec=o["pmra"] * u.mas / u.yr, pm_dec=o["pmdec"] * u.mas / u.yr,
                 obstime=Time(o["epoca"], format="jyear"), frame="icrs")
    hoy = c.apply_space_motion(new_obstime=Time(fecha))
    print(f"{clave}: catálogo {o['ra']:.4f}, {o['dec']:.4f} -> {fecha}: {hoy.ra.deg:.4f}, {hoy.dec.deg:.4f}")
    return SkyCoord(ra=hoy.ra, dec=hoy.dec, frame="icrs")

coord = posicion_actual("barnard")

In [ ]:
import os
import time
import urllib.request
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astroquery.ipac.irsa import Irsa

# 1. Definición de objetivos y coordenadas (según la guía del proyecto)
OBJETIVOS = {
    "r_dra":    dict(ra=248.1675,  dec=66.7550,   pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "rw_aur":   dict(ra=76.95642,  dec=30.40142,  pmra=0,     pmdec=0,     epoca=2000.0, size=0.1),
    "barnard":  dict(ra=269.45208, dec=4.69339,   pmra=-802,  pmdec=10362, epoca=2000.0, size=0.1),
    "wise0855": dict(ra=133.79513, dec=-7.24515,  pmra=-8120, pmdec=680,   epoca=2010.5, size=0.1),
    "eps_ind":  dict(ra=331.0438,  dec=-56.7827,  pmra=3960,  pmdec=-2540, epoca=2000.0, size=0.2),
}

def posicion_actual(clave, fecha="2026-01-01"):
    o = OBJETIVOS[clave]
    c = SkyCoord(ra=o["ra"] * u.deg, dec=o["dec"] * u.deg,
                 pm_ra_cosdec=o["pmra"] * u.mas / u.yr, 
                 pm_dec=o["pmdec"] * u.mas / u.yr,
                 obstime=Time(o["epoca"], format="jyear"), frame="icrs")
    hoy = c.apply_space_motion(new_obstime=Time(fecha))
    return SkyCoord(ra=hoy.ra, dec=hoy.dec, frame="icrs")

# 2. Ruta local dentro de tu proyecto Planet X
BASE = "./data/spherex_cache"

def descargar_objetivo(nombre, coord, size=0.1, workers=3):
    carpeta = os.path.join(BASE, nombre)
    os.makedirs(carpeta, exist_ok=True)

    print(f"Consultando IRSA para {nombre}...")
    try:
        res = Irsa.query_sia(pos=(coord, 1 * u.arcsec),
                             collection=["spherex_dr1", "spherex_qr3"])
    except Exception as e:
        print(f"Error al consultar IRSA para {nombre}: {e}")
        return

    t = res.to_table() if hasattr(res, "to_table") else res
    if len(t) == 0:
        print(f"No se encontraron registros en IRSA para {nombre}")
        return
        
    t.sort("t_min")

    filas = {}
    for r in t:
        obs = str(r["obs_id"]).replace("/", "_")
        filas[obs] = dict(  # Deduplicar si está en ambas colecciones
            archivo=f"{obs}.fits",
            url=f"{r['access_url']}?center={coord.ra.deg},{coord.dec.deg}d&size={size}",
            mjd=float(r["t_min"]),
            detector=str(r["energy_bandpassname"]),
            coleccion=str(r["obs_collection"]))
            
    filas = list(filas.values())
    pd.DataFrame(filas).to_csv(os.path.join(carpeta, "manifest.csv"), index=False)

    def bajar(f):
        destino = os.path.join(carpeta, f['archivo'])
        if os.path.exists(destino):
            return True
        for k in range(5):
            try:
                urllib.request.urlretrieve(f["url"], destino + ".tmp")
                os.replace(destino + ".tmp", destino)
                return True
            except Exception:
                time.sleep(5 * (k + 1))
        return False

    print(f"Descargando {len(filas)} archivos para {nombre}...")
    with ThreadPoolExecutor(workers) as ex:
        ok = list(ex.map(bajar, filas))
    print(f"¡Completado! {nombre}: {sum(ok)} de {len(filas)} descargados en {carpeta}")

# 3. Ejecución local para todos los objetivos del reto
for clave, o in OBJETIVOS.items():
    coord = posicion_actual(clave)
    descargar_objetivo(clave, coord, size=o["size"])